In [ ]:
"""
Descriptive tables by suicide outcome (mean and SD for numeric, n and percent for categorical)

Creates:
  1) short_descriptives_by_suicide.csv
  2) full_descriptives_by_suicide.csv

Optional:
  Add p values (Welch t test for numeric, chi square for categorical) by setting INCLUDE_P_VALUES = True

Input file:
  /mnt/data/mcs_sh_sample_Feb_28.csv
"""

from __future__ import annotations

import math
import numpy as np
import pandas as pd

try:
    from scipy import stats
except Exception:
    stats = None


# =========================
# Settings
# =========================
IN_PATH = "../data/mcs_sh_sample_Feb_28.csv"
GROUP_COL = "suicide_17y"  # expects two groups like yes and no

OUT_SHORT = "short_descriptives_by_suicide.csv"
OUT_FULL = "full_descriptives_by_suicide.csv"

INCLUDE_P_VALUES = False  # set True if you want p values
ALPHA = 0.05


# =========================
# Variable lists
# =========================
SHORT_VARS = [
    "sex",
    "mEthnicity",
    "FHYPER",
    "FCONDUCT",
    "FPEER",
    "FPROSOC",
    "FEMOTION",
    "SelfEsteem",
    "Depression",
    "cog_word_activity",
    "cog_decision_making",
    "cog_risk_adjustment",
    "cog_risk_taking",
]

FULL_VARS = [
    "sex", "mAge", "mEdu", "mEmploy", "mReligion", "mEthnicity",
    "FHYPER", "FCONDUCT", "FPEER", "FPROSOC", "FEMOTION",
    "child_alcohol", "child_cannabis", "mAlcohol", "fAlcohol",
    "child_ADHD", "child_autism", "child_ilness", "child_specneeds",
    "mNeurotic", "mConscienc", "mOpenness", "mAgree", "mExtravert",
    "fNeurotic", "fConscienc", "fOpenness", "fAgree", "fExtravert",
    "mKessler", "fKessler", "mDepression", "fDepression",
    "sexualassault", "banghead", "knockedout", "BMI", "Obesity",
    "cog_word_activity", "cog_decision_making", "cog_risk_adjustment", "cog_risk_taking",
    "mAlcohol_binary", "fAlcohol_binary",
    "mean_acc_24h", "mean_acc_5to9",
    "m5_hour_start", "m5_mean_acc",
    "l5_hour_start", "l5_mean_acc",
    "mvpa_acc_5sec", "mvpa_acc_1min", "mvpa_acc_5min",
    "mvpa_bout1", "mvpa_bout5", "mvpa_bout10",
    "SelfEsteem", "Depression",
]

# Variables that should be treated as categorical even if stored as numbers
FORCE_CATEGORICAL = {
    "sex", "mEthnicity", "mReligion", "mEdu", "mEmploy",
    "child_alcohol", "child_cannabis", "child_ADHD", "child_autism",
    "child_ilness", "child_specneeds", "sexualassault", "banghead", "knockedout",
    "Obesity", "mAlcohol_binary", "fAlcohol_binary",
}


# =========================
# Helpers
# =========================
def is_missing(x) -> bool:
    return pd.isna(x)


def safe_to_numeric(series: pd.Series) -> pd.Series:
    """Try convert to numeric; non convertibles become NaN."""
    return pd.to_numeric(series, errors="coerce")


def should_be_categorical(df: pd.DataFrame, col: str) -> bool:
    if col in FORCE_CATEGORICAL:
        return True

    s = df[col]
    if pd.api.types.is_object_dtype(s) or pd.api.types.is_bool_dtype(s) or pd.api.types.is_categorical_dtype(s):
        return True

    # Heuristic for small number of unique values
    s_num = safe_to_numeric(s)
    nunique = s_num.dropna().nunique()
    if nunique <= 9:
        return True

    return False


def format_mean_sd(mean: float, sd: float) -> str:
    if mean is None or sd is None or (isinstance(mean, float) and math.isnan(mean)) or (isinstance(sd, float) and math.isnan(sd)):
        return ""
    return f"{mean:.3f} ({sd:.3f})"


def format_n_pct(n: int, pct: float) -> str:
    if pct is None or (isinstance(pct, float) and math.isnan(pct)):
        return f"{n}"
    return f"{n} ({pct:.1f}%)"


def welch_t_pvalue(x: np.ndarray, y: np.ndarray) -> float | None:
    if stats is None:
        return None
    if len(x) < 2 or len(y) < 2:
        return None
    try:
        res = stats.ttest_ind(x, y, equal_var=False, nan_policy="omit")
        return float(res.pvalue)
    except Exception:
        return None


def chi2_pvalue(table: np.ndarray) -> float | None:
    if stats is None:
        return None
    try:
        chi2, p, dof, exp = stats.chi2_contingency(table)
        return float(p)
    except Exception:
        return None


def make_descriptive_table(
    df: pd.DataFrame,
    group_col: str,
    variables: list[str],
    include_p: bool = False,
) -> pd.DataFrame:
    # Keep only rows with non missing group
    df = df.loc[~df[group_col].isna()].copy()

    groups = list(pd.unique(df[group_col]))
    if len(groups) != 2:
        raise ValueError(f"{group_col} must have exactly 2 groups, found: {groups}")

    g0, g1 = groups[0], groups[1]

    rows = []

    for col in variables:
        if col not in df.columns:
            rows.append(
                {
                    "Variable": col,
                    "Level": "missing_column",
                    str(g0): "",
                    str(g1): "",
                    "Missing_" + str(g0): "",
                    "Missing_" + str(g1): "",
                    "p_value": "" if include_p else None,
                }
            )
            continue

        col_is_cat = should_be_categorical(df, col)

        s = df[col]
        s0 = df.loc[df[group_col] == g0, col]
        s1 = df.loc[df[group_col] == g1, col]

        miss0 = int(s0.isna().sum())
        miss1 = int(s1.isna().sum())

        if col_is_cat:
            # Treat as categorical: n and percent within group
            levels = pd.Series(pd.unique(s.dropna())).sort_values(kind="mergesort").tolist()

            # Build contingency for chi square on full levels (drop NaN)
            p_val = None
            if include_p:
                # rows: levels, cols: groups
                counts0 = []
                counts1 = []
                for lv in levels:
                    counts0.append(int((s0 == lv).sum()))
                    counts1.append(int((s1 == lv).sum()))
                cont = np.array([counts0, counts1]).T  # shape: nlevels x 2
                if cont.size > 0:
                    p_val = chi2_pvalue(cont)

            denom0 = max(1, int(s0.notna().sum()))
            denom1 = max(1, int(s1.notna().sum()))

            for i, lv in enumerate(levels):
                n0 = int((s0 == lv).sum())
                n1 = int((s1 == lv).sum())
                pct0 = 100.0 * n0 / denom0 if denom0 else np.nan
                pct1 = 100.0 * n1 / denom1 if denom1 else np.nan

                rows.append(
                    {
                        "Variable": col if i == 0 else "",
                        "Level": str(lv),
                        str(g0): format_n_pct(n0, pct0),
                        str(g1): format_n_pct(n1, pct1),
                        "Missing_" + str(g0): miss0 if i == 0 else "",
                        "Missing_" + str(g1): miss1 if i == 0 else "",
                        "p_value": (f"{p_val:.4g}" if (include_p and p_val is not None and i == 0) else "") if include_p else None,
                    }
                )

        else:
            # Numeric: mean and SD
            x0 = safe_to_numeric(s0).dropna().to_numpy(dtype=float)
            x1 = safe_to_numeric(s1).dropna().to_numpy(dtype=float)

            mean0 = float(np.mean(x0)) if len(x0) else np.nan
            sd0 = float(np.std(x0, ddof=1)) if len(x0) > 1 else np.nan

            mean1 = float(np.mean(x1)) if len(x1) else np.nan
            sd1 = float(np.std(x1, ddof=1)) if len(x1) > 1 else np.nan

            p_val = None
            if include_p:
                p_val = welch_t_pvalue(x0, x1)

            rows.append(
                {
                    "Variable": col,
                    "Level": "",
                    str(g0): format_mean_sd(mean0, sd0),
                    str(g1): format_mean_sd(mean1, sd1),
                    "Missing_" + str(g0): miss0,
                    "Missing_" + str(g1): miss1,
                    "p_value": f"{p_val:.4g}" if (include_p and p_val is not None) else "" if include_p else None,
                }
            )

    out = pd.DataFrame(rows)

    # If p values not requested, drop that column cleanly
    if not include_p and "p_value" in out.columns:
        out = out.drop(columns=["p_value"])

    return out


# =========================
# Main
# =========================
def main() -> None:
    df = pd.read_csv(IN_PATH)
    df.columns = df.columns.str.strip()

    # =========================
    # Fix structural missingness (alcohol / cannabis as categorical)
    # =========================

    for var in ["child_alcohol", "child_cannabis"]:
        if var in df.columns:
            df[var] = df[var].fillna("none")
    # Keep only requested variables that exist, but keep placeholders for missing columns too
    short_tbl = make_descriptive_table(df, GROUP_COL, SHORT_VARS, include_p=INCLUDE_P_VALUES)
    full_tbl = make_descriptive_table(df, GROUP_COL, FULL_VARS, include_p=INCLUDE_P_VALUES)

    short_tbl.to_csv(OUT_SHORT, index=False)
    full_tbl.to_csv(OUT_FULL, index=False)

    print("Saved:")
    print("  ", OUT_SHORT)
    print("  ", OUT_FULL)


if __name__ == "__main__":
    main()

In [ ]:
# =========================
# Compare actigraphy vs no actigraphy (only variables used for short descriptive table)
# =========================

OUT_ACTIG_SHORT = "short_descriptives_by_actigraphy.csv"

# 1. Create actigraphy indicator
df = pd.read_csv(IN_PATH)
df.columns = df.columns.str.strip()

# =========================
# Fix structural missingness (alcohol / cannabis as categorical)
# =========================

for var in ["child_alcohol", "child_cannabis"]:
    if var in df.columns:
        df[var] = df[var].fillna("none")

ACTIGRAPHY_VARS = [
    "mean_acc_24h", "mean_acc_5to9",
    "m5_hour_start", "m5_mean_acc",
    "l5_hour_start", "l5_mean_acc",
    "mvpa_acc_5sec", "mvpa_acc_1min", "mvpa_acc_5min",
    "mvpa_bout1", "mvpa_bout5", "mvpa_bout10",
]

available_cols = [c for c in ACTIGRAPHY_VARS if c in df.columns]

df["actigraphy_available"] = np.where(
    df[available_cols].notna().any(axis=1),
    "Actigraphy",
    "No actigraphy"
)
def add_fdr_correction(df: pd.DataFrame, p_col: str = "p_value") -> pd.DataFrame:
    df = df.copy()

    pvals = pd.to_numeric(df[p_col], errors="coerce")
    valid_idx = pvals.notna()
    pvals_valid = pvals[valid_idx].values

    if len(pvals_valid) == 0:
        df["q_value"] = ""
        return df

    n = len(pvals_valid)
    order = np.argsort(pvals_valid)
    ranked = pvals_valid[order]

    qvals = np.empty(n)
    min_q = 1.0

    for i in range(n - 1, -1, -1):
        rank = i + 1
        q = ranked[i] * n / rank
        min_q = min(min_q, q)
        qvals[i] = min_q

    qvals_correct = np.empty(n)
    qvals_correct[order] = qvals

    df["q_value"] = ""
    df.loc[valid_idx, "q_value"] = [f"{min(q, 1.0):.4g}" for q in qvals_correct]

    return df
    
# 2. Run comparison using your existing logic
actig_short_tbl = make_descriptive_table(
    df=df,
    group_col="actigraphy_available",
    variables=SHORT_VARS,
    include_p=True
)

# 3. Add FDR correction
actig_short_tbl = add_fdr_correction(actig_short_tbl, p_col="p_value")

# 4. Save
actig_short_tbl.to_csv(OUT_ACTIG_SHORT, index=False)

print("Saved:", OUT_ACTIG_SHORT)
print("\nGroup sizes:")
print(df["actigraphy_available"].value_counts())

In [ ]:
# =========================
# Compare actigraphy vs no actigraphy (all non-actigraphy variables in FULL_VARS)
# =========================

OUT_ACTIG_FULL = "full_descriptives_by_actigraphy.csv"

# Variables to compare:
# use all variables from FULL_VARS except the actigraphy variables that define wear status
FULL_VARS_NO_ACTIG = [v for v in FULL_VARS if v not in ACTIGRAPHY_VARS]

actig_full_tbl = make_descriptive_table(
    df=df,
    group_col="actigraphy_available",
    variables=FULL_VARS_NO_ACTIG,
    include_p=True
)

actig_full_tbl = add_fdr_correction(actig_full_tbl, p_col="p_value")

actig_full_tbl.to_csv(OUT_ACTIG_FULL, index=False)

print("Saved:", OUT_ACTIG_FULL)

# Optional: print only variables significant after FDR
sig_rows = actig_full_tbl.loc[
    pd.to_numeric(actig_full_tbl["q_value"], errors="coerce") < 0.05,
    ["Variable", "Level", "p_value", "q_value"]
]

print("\nVariables significant after FDR:")
if len(sig_rows) == 0:
    print("None")
else:
    print(sig_rows.to_string(index=False))

In [ ]:
# =========================
# Compare actigraphy vs no actigraphy (SHORT_VARS only)
# =========================

OUT_ACTIG_SHORT = "short_descriptives_by_actigraphy.csv"

SHORT_VARS_NO_ACTIG = [v for v in SHORT_VARS if v not in ACTIGRAPHY_VARS]

actig_short_tbl = make_descriptive_table(
    df=df,
    group_col="actigraphy_available",
    variables=SHORT_VARS_NO_ACTIG,
    include_p=True
)

actig_short_tbl = add_fdr_correction(actig_short_tbl, p_col="p_value")

actig_short_tbl.to_csv(OUT_ACTIG_SHORT, index=False)

print("Saved:", OUT_ACTIG_SHORT)

sig_rows = actig_short_tbl.loc[
    pd.to_numeric(actig_short_tbl["q_value"], errors="coerce") < 0.05,
    ["Variable", "Level", "p_value", "q_value"]
]

print("\nVariables significant after FDR:")
print("None" if len(sig_rows) == 0 else sig_rows.to_string(index=False))